In [1]:
# Importando as bibliotecas necessárias
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
import os

# Configurando a seed para reproducibilidade
random.seed(42)
np.random.seed(42)

print("Bibliotecas importadas com sucesso!")

Bibliotecas importadas com sucesso!


In [2]:
# Definindo parâmetros
num_linhas = 5500  # Um pouco mais que 5000 para garantir
data_inicio = datetime(2023, 1, 1)
data_fim = datetime(2024, 12, 31)

# Listas de dados fictícios para dar realismo
categorias_produtos = {
    'Eletrônicos': ['Smartphone', 'Fone de Ouvido', 'Carregador', 'Tablet', 'Smartwatch', 'Caixa de Som', 'Mouse', 'Teclado'],
    'Roupas': ['Camiseta', 'Calça Jeans', 'Jaqueta', 'Vestido', 'Meias', 'Tênis', 'Boné', 'Casaco'],
    'Livros': ['Ficção Científica', 'Romance', 'Autoajuda', 'Programação em Python', 'Data Science', 'História', 'Biografia', 'Aventura'],
    'Casa e Cozinha': ['Liquidificador', 'Jogo de Panelas', 'Talheres', 'Cama', 'Toalha', 'Cortina', 'Abajur', 'Cafeteira'],
    'Esportes': ['Bola de Futebol', 'Luva de Boxe', 'Tapete de Yoga', 'Halter', 'Corda de Pular', 'Garrafa Térmica', 'Mochila', 'Tênis de Corrida']
}

# Função para gerar um valor unitário aleatório baseado na categoria
def gerar_valor_unitario(categoria):
    if categoria == 'Eletrônicos':
        return round(np.random.uniform(50, 2000), 2)
    elif categoria == 'Roupas':
        return round(np.random.uniform(20, 500), 2)
    elif categoria == 'Livros':
        return round(np.random.uniform(15, 150), 2)
    elif categoria == 'Casa e Cozinha':
        return round(np.random.uniform(10, 800), 2)
    else: # Esportes
        return round(np.random.uniform(25, 600), 2)

# Gerando os dados
dados = []
ids_cliente_unicos = random.sample(range(1000, 10000), 300) # 300 clientes únicos

for i in range(1, num_linhas + 1):
    id_transacao = f"TXN{str(i).zfill(6)}"
    
    # Data aleatória entre 2023 e 2024
    dias_aleatorios = random.randint(0, (data_fim - data_inicio).days)
    data_venda = data_inicio + timedelta(days=dias_aleatorios)
    
    id_cliente = random.choice(ids_cliente_unicos)
    categoria = random.choice(list(categorias_produtos.keys()))
    nome_produto = random.choice(categorias_produtos[categoria])
    valor_unitario = gerar_valor_unitario(categoria)
    
    # A quantidade, às vezes, pode ser zero para simular erro
    if random.random() < 0.01: # 1% de chance de ser zero
        quantidade = 0
    else:
        quantidade = random.randint(1, 5)
    
    dados.append([id_transacao, data_venda, id_cliente, nome_produto, categoria, valor_unitario, quantidade])

# Criando o DataFrame
df_raw = pd.DataFrame(dados, columns=['ID_Transacao', 'Data_Venda', 'ID_Cliente', 'Nome_Produto', 'Categoria_Produto', 'Valor_Unitario', 'Quantidade'])

print(f"DataFrame gerado com {len(df_raw)} linhas e {len(df_raw.columns)} colunas.")
df_raw.head() # Mostra as primeiras linhas

DataFrame gerado com 5500 linhas e 7 colunas.


,ID_Transacao,Data_Venda,ID_Cliente,Nome_Produto,Categoria_Produto,Valor_Unitario,Quantidade
0,TXN000001,2023-05-18,9005,Fone de Ouvido,Eletrônicos,780.35,2
1,TXN000002,2023-10-06,2403,Halter,Esportes,571.66,2
2,TXN000003,2024-12-04,9005,Tênis de Corrida,Esportes,445.90,1
3,TXN000004,2023-04-05,8260,Ficção Científica,Livros,95.82,0
4,TXN000005,2023-05-14,6310,Casaco,Roupas,94.89,4


In [3]:
# Introduzindo valores nulos propositalmente
# Em 2% das linhas, vamos colocar NaN na coluna 'ID_Cliente'
indices_nulos_cliente = random.sample(range(len(df_raw)), int(len(df_raw) * 0.02))
df_raw.loc[indices_nulos_cliente, 'ID_Cliente'] = np.nan

# Em 1% das linhas, vamos colocar NaN na coluna 'Valor_Unitario'
indices_nulos_valor = random.sample(range(len(df_raw)), int(len(df_raw) * 0.01))
df_raw.loc[indices_nulos_valor, 'Valor_Unitario'] = np.nan

# Introduzindo linhas duplicadas (vamos duplicar as primeiras 10 linhas)
linhas_duplicadas = df_raw.iloc[:10].copy()
df_raw = pd.concat([df_raw, linhas_duplicadas], ignore_index=True)

print("Sujeira adicionada com sucesso!")
print(f"Valores nulos por coluna:\n{df_raw.isnull().sum()}")
print(f"\nNúmero de linhas duplicadas: {df_raw.duplicated().sum()}")

Sujeira adicionada com sucesso!
Valores nulos por coluna:
ID_Transacao           0
Data_Venda             0
ID_Cliente           110
Nome_Produto           0
Categoria_Produto      0
Valor_Unitario        55
Quantidade             0
dtype: int64

Número de linhas duplicadas: 10


In [4]:
# Salvando o dataset bruto (com sujeira)
caminho_arquivo_bruto = os.path.join('..', 'data', 'ecom_data_raw.csv')
df_raw.to_csv(caminho_arquivo_bruto, index=False)
print(f"Dataset bruto salvo em: {caminho_arquivo_bruto}")

Dataset bruto salvo em: ..\data\ecom_data_raw.csv


In [5]:
print(f"Linhas antes de remover duplicatas: {len(df_raw)}")
df_clean = df_raw.drop_duplicates()
print(f"Linhas depois de remover duplicatas: {len(df_clean)}")

Linhas antes de remover duplicatas: 5510
Linhas depois de remover duplicatas: 5500


In [6]:
df_clean = df_clean.copy()

# Tratar nulos em 'ID_Cliente': como não podemos ter uma venda sem cliente, vamos remover essas linhas.
linhas_antes = len(df_clean)
df_clean.dropna(subset=['ID_Cliente'], inplace=True)
print(f"Linhas removidas por falta de ID_Cliente: {linhas_antes - len(df_clean)}")

# Tratar nulos em 'Valor_Unitario': podemos substituir pela média da categoria do produto.
# Primeiro, calculamos a média do valor unitário por categoria (ignorando os nulos que ainda temos)
media_valor_por_categoria = df_clean.groupby('Categoria_Produto')['Valor_Unitario'].mean()

# Função para preencher o valor nulo com a média da sua categoria
def preencher_valor_nulo(row):
    if pd.isna(row['Valor_Unitario']):
        return media_valor_por_categoria[row['Categoria_Produto']]
    else:
        return row['Valor_Unitario']

df_clean['Valor_Unitario'] = df_clean.apply(preencher_valor_nulo, axis=1)

# Verificamos se ainda há nulos
print(f"Valores nulos após tratamento:\n{df_clean.isnull().sum()}")

Linhas removidas por falta de ID_Cliente: 110
Valores nulos após tratamento:
ID_Transacao         0
Data_Venda           0
ID_Cliente           0
Nome_Produto         0
Categoria_Produto    0
Valor_Unitario       0
Quantidade           0
dtype: int64


In [7]:
# Padronizar Data_Venda para datetime (pode ser que já esteja, mas garantimos)
df_clean['Data_Venda'] = pd.to_datetime(df_clean['Data_Venda'])

# Padronizar valores monetários (já estão como float, mas podemos garantir duas casas decimais)
df_clean['Valor_Unitario'] = df_clean['Valor_Unitario'].round(2)

# Padronizar IDs como string
df_clean['ID_Transacao'] = df_clean['ID_Transacao'].astype(str)
df_clean['ID_Cliente'] = df_clean['ID_Cliente'].astype(int).astype(str) # Converte de float para int e depois string

# Padronizar nomes de produtos e categorias (tirar espaços extras, capitalizar)
df_clean['Nome_Produto'] = df_clean['Nome_Produto'].str.strip().str.title()
df_clean['Categoria_Produto'] = df_clean['Categoria_Produto'].str.strip()

print("Formatos padronizados.")
df_clean.info() # Mostra os tipos de dados atualizados

Formatos padronizados.
<class 'pandas.core.frame.DataFrame'>
Index: 5390 entries, 0 to 5499
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   ID_Transacao       5390 non-null   object        
 1   Data_Venda         5390 non-null   datetime64[ns]
 2   ID_Cliente         5390 non-null   object        
 3   Nome_Produto       5390 non-null   object        
 4   Categoria_Produto  5390 non-null   object        
 5   Valor_Unitario     5390 non-null   float64       
 6   Quantidade         5390 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(4)
memory usage: 336.9+ KB


In [8]:
print(f"Linhas com Quantidade = 0 antes: {(df_clean['Quantidade'] == 0).sum()}")
df_clean.loc[df_clean['Quantidade'] == 0, 'Quantidade'] = 1
print(f"Linhas com Quantidade = 0 depois: {(df_clean['Quantidade'] == 0).sum()}")

Linhas com Quantidade = 0 antes: 68
Linhas com Quantidade = 0 depois: 0


In [9]:
# Salvando o dataset limpo na pasta data/
caminho_arquivo_limpo = os.path.join('..', 'data', 'ecom_data_clean.csv')
df_clean.to_csv(caminho_arquivo_limpo, index=False)
print(f"Dataset limpo salvo em: {caminho_arquivo_limpo}")

Dataset limpo salvo em: ..\data\ecom_data_clean.csv


In [10]:
import sqlite3

# Caminho para o arquivo do banco de dados (na pasta database/)
caminho_db = os.path.join('..', 'database', 'ecom.db')

# Conecta ao banco de dados (se o arquivo não existir, ele será criado)
conn = sqlite3.connect(caminho_db)

# Carrega o DataFrame limpo para uma tabela chamada 'vendas'
# Se a tabela já existir, substituímos (if_exists='replace')
df_clean.to_sql('vendas', conn, if_exists='replace', index=False)

print(f"Dados carregados com sucesso no banco SQLite: {caminho_db}")

# Vamos fazer uma consulta rápida para verificar se deu certo
query = "SELECT COUNT(*) as total_linhas FROM vendas;"
resultado = pd.read_sql_query(query, conn)
print(f"Total de linhas na tabela 'vendas': {resultado['total_linhas'][0]}")

# Fecha a conexão
conn.close()

Dados carregados com sucesso no banco SQLite: ..\database\ecom.db
Total de linhas na tabela 'vendas': 5390
